# Laser Injection/Recovery Statistics — ABSOLUTE amplitude & width version

This notebook duplicates `find_peak_statistics.ipynb` with two changes to how
lasers are injected:

1. **Absolute amplitude (bump sampled relative to local flux).** Instead of
   specifying amplitude as a percentage of the spectrum's max flux
   (`laser_amp_percent`), the injected bump height is drawn directly, in
   absolute flux units, from `[bump_min, bump_max]` and added on top of
   whatever the local flux happens to be at the injection pixel:
   `flux_injected = flux + bump`. This guarantees the bump is always
   non-negative by construction (no more skipped injections from an
   absolute target landing below the local flux).
2. **Fixed FWHM in Å.** The Gaussian's FWHM is specified directly in Å
   (`fwhm_aa = 0.28`) and converted to sigma using the spectrum's pixel scale
   (`cdelt1`, Å/pixel) — no root-finding, just the standard closed-form
   `sigma = fwhm_aa / cdelt1 / (2*sqrt(2*ln2))`.

The bump added on top of the local flux is randomized uniformly across
`[bump_min, bump_max] = [0.0, 1.0]` for each of the 1000 injections per
spectrum. Since the bump is drawn directly and added (not derived from an
absolute target minus local flux), it's always non-negative — no injections
need to be skipped.

In [ ]:
from scipy.signal import find_peaks
from astropy.io import fits
from specutils import Spectrum1D
import matplotlib.pyplot as plt
import numpy as np
from astropy import units as u
import requests
from io import BytesIO
import os
import csv

## Define Functions to inject, detect, and analyze

In [ ]:
def get_flux_wl(obj_id, filt, folder="/datax/scratch/emmay/galah_spectra"):
    """
    Reads a GALAH FITS spectrum and returns the wavelength and flux arrays.

    Parameters:
        obj_id (str): Object ID
        filt (str): Filter (e.g. 'B', 'V', 'R', 'I')
        folder (str): Path to FITS files

    Returns:
        wavelength (np.ndarray): Wavelength array (in Å, plain float array)
        flux (np.ndarray): Flux array
    """
    filename = f"{obj_id}_{filt}.fits"
    path = os.path.join(folder, filename)

    with fits.open(path) as hdul:
        header = hdul[1].header
        flux = hdul[1].data.astype(float)

    crval1 = header.get('CRVAL1')
    cdelt1 = header.get('CDELT1')
    crpix1 = header.get('CRPIX1', 1)

    npix = len(flux)
    wavelength = crval1 + (np.arange(npix) + 1 - crpix1) * cdelt1

    return wavelength, flux

In [ ]:
def measure_width_at_y(flux, wavelength, peak_idx, y_level=1):
    """Measure width of a peak at a fixed y_level using linear interpolation."""
    # Left side
    i = peak_idx
    while i > 0 and flux[i] > y_level:
        i -= 1
    if i == 0 or flux[i] > y_level:
        return None  # No valid left edge
    left = wavelength[i] + (y_level - flux[i]) / (flux[i+1] - flux[i]) * (wavelength[i+1] - wavelength[i])

    # Right side
    i = peak_idx
    while i < len(flux) - 1 and flux[i] > y_level:
        i += 1
    if i == len(flux) - 1 or flux[i] > y_level:
        return None  # No valid right edge
    right = wavelength[i-1] + (y_level - flux[i-1]) / (flux[i] - flux[i-1]) * (wavelength[i] - wavelength[i-1])

    return left, right, right - left

### Injection function — bump sampled relative to local flux + fixed FWHM (Å)

Rather than drawing an absolute target peak value and subtracting the local
flux (which can go negative if the drawn target happens to sit below the
noisy local flux), the **bump itself** is now drawn directly, uniformly from
`[bump_min, bump_max]` (default `[0.0, 1.0]`), and added on top of whatever
the local flux happens to be at the randomly chosen pixel:
`flux_injected = flux + bump`. This guarantees the bump is always
non-negative by construction — no skipped injections needed.

`fwhm_aa` is the Gaussian's FWHM in Å (e.g. `0.28`), converted to sigma via
the spectrum's pixel scale (`cdelt1`, Å/pixel):
`sigma_pixels = fwhm_aa / cdelt1 / (2*sqrt(2*ln2))`.

In [ ]:
def inject_and_plot_laser_absolute(obj_id, filt, folder="/datax/scratch/emmay/galah_spectra",
                                    fwhm_aa=0.28, bump_min=0.0, bump_max=1.0, plot=True):
    """
    Injects a Gaussian laser spike into a GALAH FITS spectrum. The bump height
    is drawn uniformly from [bump_min, bump_max] and added directly on top of
    the local flux at the randomly chosen injection pixel, so it is always
    non-negative by construction (bump only ever adds, never subtracts).

    Parameters:
        obj_id (str): Object ID
        filt (str): Filter (e.g. 'B', 'V', 'R', 'I')
        folder (str): Path to FITS files
        fwhm_aa (float): FWHM of the Gaussian in Angstroms (converted to sigma
            via the spectrum's pixel scale, cdelt1)
        bump_min, bump_max (float): Range to uniformly draw the bump height
            (absolute flux units) added on top of the local flux at the
            injection pixel.
        plot (bool): Whether to show the plot

    Returns:
        wavelength (np.ndarray): Wavelength array (Å, plain float array)
        flux_injected (np.ndarray): Flux array with injected Gaussian
        flux (np.ndarray): Original flux array
        injected_wavelength (Quantity): Wavelength of the injection center
        peak_amplitude (float): The resulting absolute peak flux value
            (local_flux[center] + bump_height)
    """
    filename = f"{obj_id}_{filt}.fits"
    path = os.path.join(folder, filename)

    with fits.open(path) as hdul:
        header = hdul[1].header
        flux = hdul[1].data.astype(float)

    # Build wavelength axis
    crval1 = header.get('CRVAL1')
    cdelt1 = header.get('CDELT1')
    crpix1 = header.get('CRPIX1', 1)

    npix = len(flux)
    wavelength = (crval1 + (np.arange(npix) + 1 - crpix1) * cdelt1) * u.AA

    center = np.random.randint(0, npix)  # random pixel index in [0, npix-1]
    bump_height = np.random.uniform(bump_min, bump_max)
    peak_amplitude = flux[center] + bump_height  # resulting absolute peak flux, for logging

    # Convert fixed FWHM (Å) to sigma in pixel units using the pixel scale
    sigma = (fwhm_aa / abs(cdelt1)) / (2 * np.sqrt(2 * np.log(2)))

    x = np.arange(npix)
    gaussian = bump_height * np.exp(-0.5 * ((x - center) / sigma) ** 2)
    flux_injected = flux + gaussian

    # Plot
    if plot:
        threshold_mask = gaussian > 1e-4
        plt.figure(figsize=(13, 5))
        plt.plot(wavelength, flux, color='blue', linewidth=0.6, label="Original Spectrum")
        plt.plot(wavelength[threshold_mask], flux_injected[threshold_mask], color='red', linewidth=1.0, label="Injected Region")
        plt.xlabel("Wavelength (Å)")
        plt.ylabel("Flux")
        plt.title(f"Injected Spectrum — {obj_id} ({filt}) — Bump = {bump_height:.3f}, Peak = {peak_amplitude:.3f}, FWHM = {fwhm_aa} Å")
        plt.legend()
        plt.tight_layout()
        plt.show()

    return wavelength.value, flux_injected, flux, wavelength[center], peak_amplitude

### Detection function — absolute height threshold + Å-based max width

This is the absolute-threshold detector: `height_threshold = 1 + height_fraction`
(an absolute flux value, not a percentage of max flux), and peak widths are
filtered directly in Å (`max_width_aa`) with no pixel-scale conversion needed,
since `measure_width_at_y` already returns widths in Å.

In [ ]:
def detect_laser_peak_with_fixed_level_width(
    wavelength,
    flux,
    height_fraction=0.2,  # absolute number, not percentage of max flux; threshold = 1 + height_fraction
    y_level=1.0,
    max_width_aa=3.0,  # max peak width in Angstroms
    injected_wavelength=None,
    plot=True
):
    flux = np.asarray(flux)
    wavelength = np.asarray(wavelength)

    # Height threshold: absolute flux level above continuum (assumed ~1)
    height_threshold = 1 + height_fraction
    peak_indices, properties = find_peaks(flux, height=height_threshold)

    if len(peak_indices) == 0:
        if plot:
            plt.figure(figsize=(12, 4))
            plt.plot(wavelength, flux, label="Full Spectrum")
            if injected_wavelength is not None:
                plt.axvline(x=float(injected_wavelength.value), color='r', linestyle='--', linewidth=2, label="Injected Laser")
            plt.xlabel("Wavelength (Å)")
            plt.ylabel("Flux")
            plt.title("Full Spectrum (No Peaks Detected)")
            plt.legend()
            plt.tight_layout()
            plt.show()
        return [], [], []

    peak_wavelengths = []
    peak_fluxes = []
    widths = []
    valid_peak_indices = []

    for i, idx in enumerate(peak_indices):
        result = measure_width_at_y(flux, wavelength, idx, y_level=y_level)
        if result is not None:
            left, right, width = result

            if width > max_width_aa:
                continue

            peak_wavelengths.append(wavelength[idx])
            peak_fluxes.append(flux[idx])
            widths.append(width)
            valid_peak_indices.append(idx)

            if plot:
                i_min = max(0, idx - 20)
                i_max = min(len(wavelength), idx + 20)

                plt.figure(figsize=(10, 4))
                plt.plot(wavelength[i_min:i_max], flux[i_min:i_max], label="Flux")
                plt.axvline(left, color='C2', linestyle='--')
                plt.axvline(right, color='C2', linestyle='--')
                plt.hlines(y_level, left, right, color='C3', linewidth=2, label=f'Width @ y={y_level}')
                plt.plot(wavelength[idx], flux[idx], 'rx', label="Peak Center")
                plt.xlabel("Wavelength (Å)")
                plt.ylabel("Flux")
                plt.title(f"Zoomed Peak at {wavelength[idx]:.2f} Å, Width = {width:.3f} Å")
                plt.legend()
                plt.tight_layout()
                plt.show()

    if plot and valid_peak_indices:
        plt.figure(figsize=(12, 4))
        plt.plot(wavelength, flux, label="Full Spectrum")
        for idx in valid_peak_indices:
            plt.plot(wavelength[idx], flux[idx], 'rx', label="Detected Peaks")

        if injected_wavelength is not None:
            plt.axvline(x=float(injected_wavelength.value), color='black', linestyle='--', linewidth=2, label="Injected Laser")

        plt.xlabel("Wavelength (Å)")
        plt.ylabel("Flux")
        plt.title("Full Spectrum with Detected Peaks")
        plt.legend()
        plt.tight_layout()
        plt.show()

    return peak_wavelengths, peak_fluxes, widths

In [ ]:
def evaluate_detection(injected_wavelengths, detected_wavelengths, threshold_angstroms=2.0):
    """
    Compare injected laser wavelengths to detected peaks and compute performance metrics.

    Parameters:
    - injected_wavelengths: list of float or Quantity
    - detected_wavelengths: list of float or Quantity
    - threshold_angstroms: float, matching window in angstroms

    Returns:
    - stats: dict with TP, FP, FN, precision, recall, F1, n_injected, n_detected
    - matched_flags: list of bool per injected wavelength
    """
    def to_angstroms(wl):
        return float(wl.to_value('angstrom')) if hasattr(wl, 'to_value') else float(wl)

    injected_wavelengths = [to_angstroms(wl) for wl in injected_wavelengths]
    detected_wavelengths = [to_angstroms(wl) for wl in detected_wavelengths]

    matched_injected = [False] * len(injected_wavelengths)
    matched_detected = [False] * len(detected_wavelengths)

    for i, inj_wl in enumerate(injected_wavelengths):
        for j, det_wl in enumerate(detected_wavelengths):
            if not matched_detected[j] and abs(det_wl - inj_wl) <= threshold_angstroms:
                matched_injected[i] = True
                matched_detected[j] = True
                break

    TP = sum(matched_injected)
    FN = len(injected_wavelengths) - TP
    FP = len(detected_wavelengths) - TP

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    stats = {
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "n_injected": len(injected_wavelengths),
        "n_detected": len(detected_wavelengths)
    }

    return stats, matched_injected

## Inject 1 laser per spectrum (sanity check)

In [ ]:
import os
import numpy as np

# Loop through all files in the folder
folder = "/datax/scratch/emmay/galah_spectra"
fits_files = [f for f in os.listdir(folder) if f.endswith(".fits")]

# Parameters
threshold_angstroms = 2.0
height_fraction = 0.15
y_level = 1.0
max_width_aa = 5.0
fwhm_aa = 0.28   # fixed Gaussian FWHM (Å)
bump_min, bump_max = 0.0, 1.0  # bump range (absolute flux units), added on top of local flux

# To store results
injected_wavelengths_all = []
injected_amplitudes_all = []
recovered_flags_all = []

for fits_file in fits_files[:10]:  # Adjust number as needed
    filename_no_ext = fits_file[:-5]
    obj_id = filename_no_ext[:15]
    filt = filename_no_ext[16:]

    try:
        wave, flux, flux_orig, injected_wl, injected_amplitude = inject_and_plot_laser_absolute(
            obj_id, filt, folder=folder,
            fwhm_aa=fwhm_aa,
            bump_min=bump_min, bump_max=bump_max,
            plot=False
        )

        # Detect peaks
        detected_wavelengths, detected_fluxes, detected_widths = detect_laser_peak_with_fixed_level_width(
            wave, flux,
            height_fraction=height_fraction,
            y_level=y_level,
            max_width_aa=max_width_aa,
            injected_wavelength=injected_wl,
            plot=False
        )

        # Evaluate detection
        stats, matched = evaluate_detection(
            injected_wavelengths=[injected_wl],
            detected_wavelengths=detected_wavelengths,
            threshold_angstroms=threshold_angstroms
        )

        # Save data for plotting later
        injected_wavelengths_all.append(injected_wl)
        injected_amplitudes_all.append(injected_amplitude)
        recovered_flags_all.append(matched[0])  # matched is list of bool, one per injection

    except Exception as e:
        # Skip file if a read/processing error occurs
        print('skipped injection')
        print(f"Error processing {fits_file}: {e}")
        continue

# Convert to numpy arrays for easier analysis
injected_wavelengths_all = np.array([w.to_value(u.Angstrom) for w in injected_wavelengths_all])
injected_amplitudes_all = np.array(injected_amplitudes_all)
recovered_flags_all = np.array(recovered_flags_all)

print(f"Injections attempted: {len(fits_files[:10])}, succeeded: {len(injected_amplitudes_all)}")

In [ ]:
np.savez(
    "single_laser_injection_results_absolute.npz",
    injected_wavelengths=injected_wavelengths_all,
    injected_amplitudes=injected_amplitudes_all,
    recovered_flags=recovered_flags_all
)

## Now inject 1000 lasers per spectrum

In [ ]:
import os
import numpy as np

# Loop through all files in the folder
folder = "/datax/scratch/emmay/galah_spectra"
fits_files = [f for f in os.listdir(folder) if f.endswith(".fits")]

# Parameters
threshold_angstroms = 2.0
height_fraction = 0.15
y_level = 1.0
max_width_aa = 5.0
fwhm_aa = 0.28   # fixed Gaussian FWHM (Å)
bump_min, bump_max = 0.0, 1.0  # bump range (absolute flux units), added on top of local flux

# To store results
injected_wavelengths_all = []
injected_amplitudes_all = []
recovered_flags_all = []

for fits_file in fits_files[:40]:  # Adjust number as needed
    filename_no_ext = fits_file[:-5]
    obj_id = filename_no_ext[:15]
    filt = filename_no_ext[16:]

    for i in range(1000):

        try:
            wave, flux, flux_orig, injected_wl, injected_amplitude = inject_and_plot_laser_absolute(
                obj_id, filt, folder=folder,
                fwhm_aa=fwhm_aa,
                bump_min=bump_min, bump_max=bump_max,
                plot=False
            )

            # Detect peaks
            detected_wavelengths, detected_fluxes, detected_widths = detect_laser_peak_with_fixed_level_width(
                wave, flux,
                height_fraction=height_fraction,
                y_level=y_level,
                max_width_aa=max_width_aa,
                injected_wavelength=injected_wl,
                plot=False
            )

            # Evaluate detection
            stats, matched = evaluate_detection(
                injected_wavelengths=[injected_wl],
                detected_wavelengths=detected_wavelengths,
                threshold_angstroms=threshold_angstroms
            )

            # Save data for plotting later
            injected_wavelengths_all.append(injected_wl)
            injected_amplitudes_all.append(injected_amplitude)
            recovered_flags_all.append(matched[0])  # matched is list of bool, one per injection

        except Exception as e:
            # Skip injection if error occurs (e.g. non-positive bump)
            print('skipped injection')
            print(f"Error processing {fits_file} (injection {i}): {e}")
            continue

# Convert to numpy arrays for easier analysis
injected_wavelengths_all = np.array([w.to_value(u.Angstrom) for w in injected_wavelengths_all])
injected_amplitudes_all = np.array(injected_amplitudes_all)
recovered_flags_all = np.array(recovered_flags_all)

print(f"Total injections succeeded: {len(injected_amplitudes_all)}")

# Now you can make:

# 1. Completeness curve: For binned amplitude values, compute fraction recovered
amplitude_bins = np.linspace(injected_amplitudes_all.min(), injected_amplitudes_all.max(), 10)
completeness = []

for i in range(len(amplitude_bins)-1):
    bin_mask = (injected_amplitudes_all >= amplitude_bins[i]) & (injected_amplitudes_all < amplitude_bins[i+1])
    if np.any(bin_mask):
        fraction_recovered = np.mean(recovered_flags_all[bin_mask])
    else:
        fraction_recovered = np.nan
    completeness.append(fraction_recovered)

completeness = np.array(completeness)
# amplitude_bins[:-1] gives bin starts for plotting

# 2. Injection scatter: wavelength vs amplitude, color by recovered_flags_all
# Store these arrays and plot later as needed

In [ ]:
np.savez(
    "thousand_laser_injection_results_absolute.npz",
    injected_wavelengths=injected_wavelengths_all,
    injected_amplitudes=injected_amplitudes_all,
    recovered_flags=recovered_flags_all
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Define bins for amplitude
n_bins = 10
amplitude_bins = np.linspace(injected_amplitudes_all.min(), injected_amplitudes_all.max(), n_bins + 1)

# Compute fraction recovered per bin
bin_centers = 0.5 * (amplitude_bins[:-1] + amplitude_bins[1:])
completeness = []

for i in range(n_bins):
    in_bin = (injected_amplitudes_all >= amplitude_bins[i]) & (injected_amplitudes_all < amplitude_bins[i+1])
    if np.any(in_bin):
        frac = np.mean(recovered_flags_all[in_bin])
    else:
        frac = np.nan
    completeness.append(frac)

# Plot
plt.figure(figsize=(8, 5))
plt.plot(bin_centers, completeness, marker='o', linestyle='-', color='darkorange')
plt.xlabel("Injected Amplitude (Absolute Peak Flux)")
plt.ylabel("Fraction Recovered")
plt.title("Completeness Curve")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Create scatter plot: wavelength vs amplitude, colored by recovery status
colors = np.where(recovered_flags_all, "green", "red")
labels = np.where(recovered_flags_all, "Recovered", "Missed")

plt.figure(figsize=(9, 6))
scatter = plt.scatter(
    injected_wavelengths_all,
    injected_amplitudes_all,
    c=recovered_flags_all,
    cmap="bwr",
    edgecolor="k",
    alpha=0.3,
)

plt.xlabel("Injected Wavelength (Å)")
plt.ylabel("Injected Amplitude (Absolute Peak Flux)")
plt.title("Injection Recovery: Wavelength vs Amplitude")
plt.grid(True)
plt.tight_layout()

# Custom legend
from matplotlib.patches import Patch
plt.legend(handles=[
    Patch(color="red", label="Not Recovered"),
    Patch(color="blue", label="Recovered")
])
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np

# Assuming these arrays exist:
# injected_wavelengths_all, injected_amplitudes_all, recovered_flags_all

# Your 4 wavelength ranges
wavelength_ranges = [
    (4718, 4903),
    (5649, 5873),
    (6481, 6739),
    (7590, 7890)
]

fig, axs = plt.subplots(2, 2, figsize=(14, 10), sharey=True)
axs = axs.flatten()

for i, (wmin, wmax) in enumerate(wavelength_ranges):
    ax = axs[i]
    # Mask for points in this wavelength range
    mask = (injected_wavelengths_all >= wmin) & (injected_wavelengths_all <= wmax)

    # Scatter plot with recovery colored: blue = recovered (True), red = missed (False)
    scatter = ax.scatter(
        injected_wavelengths_all[mask],
        injected_amplitudes_all[mask],
        c=recovered_flags_all[mask],
        cmap='bwr_r',
        edgecolor='k',
        alpha=0.6,
        vmin=0, vmax=1  # Ensure colors correspond to False=0 (red) and True=1 (blue)
    )

    ax.set_xlim(wmin, wmax)
    ax.set_xlabel("Injected Wavelength (Å)")
    if i % 2 == 0:
        ax.set_ylabel("Injected Amplitude (Absolute Peak Flux)")
    ax.set_title(f"Wavelength Range: {wmin}–{wmax} Å")
    ax.grid(True)

# Custom legend (outside loop)
fig.legend(
    handles=[
        Patch(color='red', label='Not Recovered'),
        Patch(color='blue', label='Recovered')
    ],
    loc='upper right'
)

plt.tight_layout()
plt.show()